In [66]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%env OPENAI_API_KEY=""
%env HUGGINGFACE_TOKEN = ""
                   
if not 'dirguard' in locals():
  %cd ..
  dirguard = True

%aimport vivabench
%aimport vivabench.util
%aimport vivabench.util
%aimport vivabench.ontology.concepts

from vivabench.ontology import concepts
from vivabench.ontology.concepts import *
from vivabench.util import *
from vivabench.examiner import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: OPENAI_API_KEY=""
env: HUGGINGFACE_TOKEN=""


In [9]:
# Setup stuff
import json
with open("cases/msd_raw/msd_1.json", "r") as f:
    g = json.loads(f.read())
sample_case = ClinicalCase.from_dict(g)

examiner_model = "gpt-4-1106-preview"
agent_model = "gpt-4-1106-preview"

The backbone is the Examiner class. It loads up a sample case, and provides step-wise tests for an agent.

In [8]:
examiner = Examiner(sample_case, examiner_model)
print(examiner.initial_prompt())

A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates.
You are currently reviewing this patient. What information would you like to seek from your history and examination?


In [51]:
with open('vivabench/prompts/agent.prompt', 'r') as f:
    agent_prompt_template = f.read()

In [52]:
i = 0
limit = 10
max_length = 512

In [53]:
case_prompt = examiner.initial_prompt()

In [54]:
agent_history = []
examiner_history = []
chat_history = ""

examiner = Examiner(sample_case, examiner_model)
while i < limit:

    agent_prompt = str_to_msgs(agent_prompt_template.format(
          case_prompt=case_prompt,
          chat_history=chat_history
      ))

    agent_response = chatgpt_decode(agent_prompt, model_name=agent_model, max_length=max_length)    
    
    agent_history.append(agent_response)
    
    agent_response = agent_response.split('Response to examiner:')[-1].strip()

    agent_parsed = "STUDENT: %s\n" % agent_response 
    print(agent_parsed)
    chat_history += agent_parsed
    
    examiner_response = examiner.reply(agent_response)
    
    examiner_history.append(examiner_response)
    examiner_parsed = "EXAMINER: %s\n" % examiner_response
    print(examiner_parsed)
    chat_history += examiner_parsed
    
    i += 1

    if examiner.step == "termination": break

STUDENT: I would like to inquire about the nature, location, and severity of the abdominal pain, any associated symptoms such as vaginal bleeding, discharge, fever, nausea, or vomiting, her obstetric history including previous pregnancies and their outcomes, any recent trauma or intercourse, and her medical and surgical history. On examination, I would assess her vital signs, perform an abdominal examination to evaluate for tenderness, rebound, or guarding, and conduct an obstetric examination including fetal heart rate monitoring.

EXAMINER: The patient has described her abdominal pain as sharp, steady, and radiating across her lower abdomen bilaterally. She has also experienced new onset nausea and vomiting, and she has been unable to keep down any food or drink this morning. She reports feeling cold and shivering, followed by feeling warm, but did not check her temperature. There has been no vaginal bleeding.

Regarding her obstetric history, she is currently 18 weeks pregnant and t

In [58]:
# The measurable metrics currently are just F1 scores. We can probably add a bit more quantitative and qualitative analysis. 
examiner.f1_scores # primary, diagnosis, and management.

[0.3333333333333333, 0.6666666666666666, 0.6]

### Overall assessment:
AI failed to recognize that this patient is septic. Probably worth discussing. Although the management plan is appropriate. Ceftriaxone is the accurate one, but it wasn't specific enough. You also most definitely will not use gentamicin in this pregnant patient.

Q: How do we get AI to assess this?

In [64]:
for h in agent_history:
    print("%s\n======\n" % h)

Reasoning: In a pregnant patient presenting with lower abdominal pain, it is crucial to obtain a detailed obstetric history, characteristics of the pain, associated symptoms, and any factors that may indicate complications such as ectopic pregnancy, miscarriage, or preterm labor. Additionally, a thorough examination including vital signs, abdominal examination, and obstetric examination (including fetal heart rate monitoring) is necessary to assess the health of both the mother and the fetus.

Response to examiner: I would like to inquire about the nature, location, and severity of the abdominal pain, any associated symptoms such as vaginal bleeding, discharge, fever, nausea, or vomiting, her obstetric history including previous pregnancies and their outcomes, any recent trauma or intercourse, and her medical and surgical history. On examination, I would assess her vital signs, perform an abdominal examination to evaluate for tenderness, rebound, or guarding, and conduct an obstetric e

In [65]:
# With this framework, Mixtral seems to work well as an examiner as well.
examiner_model = "mistralai/Mixtral-8x7B-Instruct-v0.1"

agent_history = []
examiner_history = []
chat_history = ""

examiner = Examiner(sample_case, examiner_model)
while i < limit:

    agent_prompt = str_to_msgs(agent_prompt_template.format(
          case_prompt=case_prompt,
          chat_history=chat_history
      ))

    agent_response = chatgpt_decode(agent_prompt, model_name=agent_model, max_length=max_length)    
    
    agent_history.append(agent_response)
    
    agent_response = agent_response.split('Response to examiner:')[-1].strip()

    agent_parsed = "STUDENT: %s\n" % agent_response 
    print(agent_parsed)
    chat_history += agent_parsed
    
    examiner_response = examiner.reply(agent_response)
    
    examiner_history.append(examiner_response)
    examiner_parsed = "EXAMINER: %s\n" % examiner_response
    print(examiner_parsed)
    chat_history += examiner_parsed
    
    i += 1

    if examiner.step == "termination": break

STUDENT: I would like to inquire about the nature and location of the abdominal pain, any associated symptoms such as vaginal bleeding, discharge, fever, nausea, vomiting, urinary or bowel changes, her obstetric history including any previous complications, sexual history, and any recent trauma or interventions. On examination, I would assess her vital signs, perform an abdominal examination to assess for tenderness, rebound, or guarding, and if appropriate, a speculum or bimanual examination to assess the cervix and adnexa.

EXAMINER: Based on the information provided in the clinical scenario, you have asked about many relevant aspects of the patient's history and examination. I will provide further details based on your questions and examination steps.

The patient is a 26-year-old woman who is 18 weeks pregnant, presenting with a 3-day history of lower abdominal pain. The pain is sharp, steady, and radiating across her lower abdomen bilaterally. Last night, she developed new nausea 